In [20]:
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
import pandas as pd
import os 
# Configuration constants
MIN_POSTS_PER_USER = 2

# Plan
1. Consider just the profiles that are present in chunk_0_posts_parquet: "user of interests". From them create a dictionary for fast joining_date_lookups.
2. "posts" time filtering of two weeks. 
3. We can further reduce the users of interest by asking if it has at least two posts. 
3. "blocks", "follows", "likes" filtering based on:
   1. The event happened in the first week after the joining date. 
   2. The user is in the user of interests. 

NB: We should create our final dataset consider the posts in chunks.

# User of Interests
We're going to study the users present in "chunk_0_posts" which have a bluesky joining date.

Furthermore we're filtering out users with $\le 1$ posts. 

In [ ]:
# Load the posts data
posts_path = "../data/posting/cleaned/chunk_0_posts.parquet"
posts_table = pq.read_table(posts_path)

posts_df = posts_table.to_pandas()

print(f"Initial posts count: {len(posts_df)}")
print(f"Unique users: {posts_df['did_id'].nunique()}")

user_post_counts = posts_df.groupby('did_id').size().reset_index(name='post_count')

# Filter out users with less than MIN_POSTS_PER_USER posts.
active_posters = user_post_counts[user_post_counts['post_count'] >= MIN_POSTS_PER_USER]['did_id'].values
filtered_posts_df = posts_df[posts_df['did_id'].isin(active_posters)]

print(f"\nPosts after filtering users with <{MIN_POSTS_PER_USER} posts: {len(filtered_posts_df)}")
print(f"Active posters (≥{MIN_POSTS_PER_USER} posts): {len(active_posters)}")

Initial posts count: 4994663
Unique users: 130111

Posts after filtering users with <2 posts: 4957527
Active users (≥2 posts): 92975

Posts after filtering users with <2 posts: 4957527
Active users (≥2 posts): 92975


In [ ]:
# Load filter profiles for users of interest only
profiles_path = "../data/posting/cleaned/profiles.parquet"
profiles_table = pq.read_table(profiles_path)

# Filter to active_posters only
profiles_table = profiles_table.filter(pc.is_in(profiles_table['did_id'], pa.array(active_posters)))
user_of_interests = profiles_table.to_pandas()

print(f"Users of interests (active posters -- ≥{MIN_POSTS_PER_USER} posts -- with a join date): {len(user_of_interests)}")

Users of interests (active posters (≥2 posts) with a join date): 65380


In [15]:
# Create a fast lookup dictionary: did_id -> join_date
join_date_dict = dict(zip(
    user_of_interests['did_id'],
    user_of_interests['created_at']
))

print(f"Join date lookup dictionary created: {len(join_date_dict)} entries")

Join date lookup dictionary created: 65376 entries


## Event Filtering
We filter an event (e.g. block, follow) if:
1. It doesn't involve an user of interest.
2. It happens after one week since the joining date.

In [ ]:
# Generic function to filter event databases with multithreading
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

def filter_events(input_path, output_path, db_name, is_valid_event, num_threads=4):
    """
    Filter event databases by row groups with custom validation logic (multithreaded).
    
    Args:
        input_path (str): Path to input parquet file
        output_path (str): Path to output parquet file
        db_name (str): Name of database (for logging)
        is_valid_event (callable): Function that takes a row and returns True/False
        num_threads (int): Number of threads to use for processing row groups
    """
    print(f"Filtering {db_name} by row groups (using {num_threads} threads)...")
    
    # Open Parquet file
    pf = pq.ParquetFile(input_path)
    print(f"Total {db_name}: {pf.metadata.num_rows:,}")
    print(f"Row groups: {pf.num_row_groups}")
    
    # Get input file size
    input_size = os.path.getsize(input_path)
    print(f"Input file size: {input_size / (1024**3):.2f} GB")
    
    # Process row groups and collect filtered tables
    filtered_tables = {}
    total_rows_processed = 0
    total_rows_kept = 0
    
    def process_row_group(rg_index):
        """Process a single row group and return (index, filtered_table, rows_kept)"""
        row_group_table = pf.read_row_group(rg_index)
        rows_in_group = row_group_table.num_rows
        
        # Convert to pandas for easier filtering with custom logic
        row_group_df = row_group_table.to_pandas()
        
        # Apply custom validation function
        filtered_df = row_group_df[row_group_df.apply(is_valid_event, axis=1)]
        
        # Convert back to PyArrow table
        filtered_table = pa.Table.from_pandas(filtered_df, preserve_index=False)
        
        return rg_index, filtered_table, len(filtered_df), rows_in_group
    
    # Use ThreadPoolExecutor to process row groups in parallel
    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = {executor.submit(process_row_group, i): i for i in range(pf.num_row_groups)}
        
        for future in as_completed(futures):
            rg_index, filtered_table, rows_kept, rows_total = future.result()
            filtered_tables[rg_index] = filtered_table
            total_rows_processed += rows_total
            total_rows_kept += rows_kept
            
            print(f"✅ Row group {rg_index+1}/{pf.num_row_groups}: {rows_total:,} → {rows_kept:,} rows")
    
    # Write all filtered tables in order
    print("\nWriting filtered results...")
    writer = None
    for i in range(pf.num_row_groups):
        filtered_table = filtered_tables[i]
        
        # Initialize writer with first filtered table schema
        if writer is None:
            writer = pq.ParquetWriter(
                output_path,
                filtered_table.schema,
                compression='zstd',
                use_dictionary=True,
                write_statistics=True
            )
        
        # Write filtered row group
        if filtered_table.num_rows > 0:
            writer.write_table(filtered_table)
    
    if writer:
        writer.close()
    
    # Calculate statistics
    retention_rate = (total_rows_kept / total_rows_processed * 100) if total_rows_processed > 0 else 0
    output_size = os.path.getsize(output_path)
    size_reduction = input_size - output_size
    size_reduction_percent = (size_reduction / input_size * 100) if input_size > 0 else 0
    
    print("\n" + "="*60)
    print(f"{db_name.upper()} FILTERING SUMMARY:")
    print("="*60)
    print(f"Row groups processed: {pf.num_row_groups} (with {num_threads} threads)")
    print(f"Rows:       {total_rows_processed:,} → {total_rows_kept:,} ({retention_rate:.1f}% kept)")
    print(f"File size:  {input_size / (1024**3):.2f} GB → {output_size / (1024**3):.2f} GB")
    print(f"Reduction:  {size_reduction / (1024**3):.2f} GB ({size_reduction_percent:.1f}%)")
    print(f"Output: {output_path}")
    print("="*60)

In [22]:
# Filter blocks
def is_valid_block(row):
    """
    Keep if: at least one user is of interest AND block happened within 7 days of their joining.
    """
    did_in_interest = row['did_id'] in join_date_dict

    subject_in_interest = row['subject_id'] in join_date_dict
    
    # Must involve at least one user of interest
    if not (did_in_interest or subject_in_interest):
        return False
    
    # Check if block happened within 7 days of joining for at least one user of interest
    if did_in_interest:
        days_since_did_join = (row['created_at'] - join_date_dict[row['did_id']]).days
        if 0 <= days_since_did_join <= 7:
            return True
    
    if subject_in_interest:
        days_since_subject_join = (row['created_at'] - join_date_dict[row['subject_id']]).days
        if 0 <= days_since_subject_join <= 7:
            return True
    
    return False


# Filter blocks using the generic function
blocks_input_path = "../data/posting/cleaned/blocks.parquet"
blocks_output_path = "../data/posting/filtered/blocks_filtered.parquet"

filter_events(blocks_input_path, blocks_output_path, "blocks", is_valid_block)

Filtering blocks by row groups...
Total blocks: 120,084,926
Row groups: 978
Input file size: 1.45 GB
Processing row group 1/978... ✅ 122,880 → 839 rows
Processing row group 2/978... ✅ 122,878 → 394 rows
Processing row group 3/978... ✅ 122,880 → 194 rows
Processing row group 4/978... ✅ 122,880 → 269 rows
Processing row group 5/978... ✅ 122,880 → 150 rows
Processing row group 6/978... ✅ 122,880 → 172 rows
Processing row group 7/978... ✅ 122,880 → 146 rows
Processing row group 8/978... ✅ 122,880 → 347 rows
Processing row group 9/978... ✅ 122,880 → 273 rows
Processing row group 10/978... ✅ 122,880 → 228 rows
Processing row group 11/978... ✅ 122,880 → 203 rows
Processing row group 12/978... ✅ 122,880 → 193 rows
Processing row group 13/978... ✅ 122,880 → 283 rows
Processing row group 14/978... ✅ 122,880 → 264 rows
Processing row group 15/978... ✅ 122,880 → 147 rows
Processing row group 16/978... ✅ 122,880 → 247 rows
Processing row group 17/978... ✅ 122,880 → 271 rows
Processing row group 18/

KeyboardInterrupt: 

In [ ]:
# Load the filtered blocks
blocks_table = pq.read_table(blocks_output_path)
blocks_df = blocks_table.to_pandas()

print(f"Filtered blocks loaded: {len(blocks_df)}")
print("\nBlocks schema:")
print(blocks_table.schema)
print("\nFirst few rows:")
print(blocks_df.head())

## Merging and Processing
We want a table where for each user we have the following stuff: 
1) Join date timestamp. 
2) First post timestamp.
3) day_1_posts, day_2_posts, ..., day_n_posts.
4) TODO: Look at the block database. 
5) TODO: Look at the likes database.

In [4]:
# Inner join profiles with posts
merged_df = filtered_posts_df.merge(
    active_profiles, on='did_id', how='inner'
)
print(merged_df.head())

                        created_at  did_id                        join_date
0 2024-08-30 21:25:28.002000+00:00     318 2024-08-30 21:04:26.303000+00:00
1 2024-08-30 22:29:13.143000+00:00     318 2024-08-30 21:04:26.303000+00:00
2 2024-08-31 14:01:03.295000+00:00     318 2024-08-30 21:04:26.303000+00:00
3 2024-08-31 14:23:46.432000+00:00     318 2024-08-30 21:04:26.303000+00:00
4 2024-08-31 14:39:12.549000+00:00     318 2024-08-30 21:04:26.303000+00:00


In [5]:
# Create a user_table with "did_id", "user_join_date" and "user_first_post"
user_first_post = merged_df.groupby('did_id')['created_at'].min().reset_index()
user_first_post = user_first_post.rename(columns={'created_at': 'first_post_date'})

user_table = active_profiles[['did_id', 'join_date']].merge(user_first_post, on='did_id', how='inner')
print("User Table")
print(user_table.head())

User Table
   did_id                        join_date                  first_post_date
0     318 2024-08-30 21:04:26.303000+00:00 2024-08-30 21:25:28.002000+00:00
1    1032 2024-10-19 12:08:26.894000+00:00 2024-11-17 23:04:06.860000+00:00
2    1738 2024-11-24 22:41:30.245000+00:00 2024-11-24 22:44:22.029000+00:00
3    3316 2024-06-08 03:54:37.834000+00:00 2024-06-08 06:02:30.972000+00:00
4    3360 2024-11-17 05:41:26.862000+00:00 2024-11-17 05:44:20.897000+00:00


In [ ]:
# Calculate days since joining, filter the first month of activity, merge them to user_table

merged_df['days_since_join'] = (
    (merged_df['created_at'] - merged_df['join_date']).dt.total_seconds() / (24 * 3600)
).round().astype(int)

first_month_posts = merged_df[
    (merged_df['days_since_join'] >= 0) & 
    (merged_df['days_since_join'] <= 15)
]

daily_post_counts = first_month_posts.groupby(['did_id', 'days_since_join']).size().reset_index(name='post_count')

# Pivot
time_series_wide = daily_post_counts.pivot_table(
    index='did_id', 
    columns='days_since_join', 
    values='post_count', 
    fill_value=0
).reset_index()

# Rename the day-posts columns
time_series_wide.columns = ['did_id'] + [f'day_{int(col)}_posts' for col in time_series_wide.columns[1:]]

# Merge into final table
user_table = user_table.merge(time_series_wide, on='did_id', how='left')

print(user_table.head())

   did_id                        join_date                  first_post_date  \
0     318 2024-08-30 21:04:26.303000+00:00 2024-08-30 21:25:28.002000+00:00   
1    1032 2024-10-19 12:08:26.894000+00:00 2024-11-17 23:04:06.860000+00:00   
2    1738 2024-11-24 22:41:30.245000+00:00 2024-11-24 22:44:22.029000+00:00   
3    3316 2024-06-08 03:54:37.834000+00:00 2024-06-08 06:02:30.972000+00:00   
4    3360 2024-11-17 05:41:26.862000+00:00 2024-11-17 05:44:20.897000+00:00   

   day_0_posts  day_1_posts  day_2_posts  day_3_posts  day_4_posts  \
0          2.0          5.0          4.0          4.0          3.0   
1          0.0          0.0          0.0          0.0          0.0   
2         19.0          0.0          0.0          0.0          0.0   
3          5.0          0.0          0.0          1.0          0.0   
4          1.0         18.0         11.0          4.0         20.0   

   day_5_posts  day_6_posts  ...  day_21_posts  day_22_posts  day_23_posts  \
0          3.0          0.

# Saving 

In [7]:
# Save the processed data for future use
output_path = "../data/posting/processed/user_activity.parquet"
user_table.to_parquet(output_path, index=False)
print(f"\nData saved to: {output_path}")


Data saved to: ../data/posting/processed/user_activity.parquet
